# LPM_transplant — 02_host_donor_pixel_quantification

**Feeds:** Fig 5n

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


            # 02 | Host/Donor Pixel Quantification

            ## Notebook Scope

            This notebook reviews the pooled ratio thresholds, the host/donor class balance, and the first-pass FOXF1 summaries stratified by pixel identity.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter1d
from scipy.stats import dunnett
from skimage import measure

from scripts import run_pixel_level_quantification as rpq
from scripts import transplant_quantification_helpers as tqh

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

            ## Paths And Parameters

            The default workflow reads outputs written by notebook `01`.
            Set `RERUN_PIPELINE=True` if you want this notebook to rerun the current quantification stage directly.
            

In [ ]:
DATA_DIR = ROOT / "data"
PLANE_METRICS_OUTPUT = ROOT / "results" / "tables" / "01_mask_and_plane_metrics.tsv"
NORMALIZATION_OUTPUT = ROOT / "results" / "tables" / "01b_condition_channel_normalization.tsv"
NORMALIZATION_HIST_OUTPUT = ROOT / "results" / "tables" / "01b_condition_channel_normalization_histograms.tsv"
THRESHOLD_OUTPUT = ROOT / "results" / "tables" / "02_ratio_thresholds.tsv"
CLASS_SUMMARY_OUTPUT = ROOT / "results" / "tables" / "02_pixel_class_summary.tsv"
FOXF1_IDENTITY_OUTPUT = ROOT / "results" / "tables" / "02_foxf1_positive_identity_summary.tsv"
PIXEL_SAMPLE_OUTPUT = ROOT / "results" / "tables" / "02_sampled_pixel_profiles.tsv"
FIGURE_DIR = ROOT / "results" / "figures"
HOST_FOXF1_BAR_PNG = FIGURE_DIR / "02_foxf1_positive_host_fraction_barplot.png"
HOST_FOXF1_BAR_SVG = FIGURE_DIR / "02_foxf1_positive_host_fraction_barplot.svg"
HOST_FOXF1_BAR_PDF = FIGURE_DIR / "02_foxf1_positive_host_fraction_barplot.pdf"

RERUN_PIPELINE = False
THRESHOLD_REFERENCE_CONDITION = "ctrl"
HOST_THRESHOLD_SCOPE = "condition"
DONOR_THRESHOLD_SCOPE = "condition"
FOXF1_THRESHOLD_SCOPE = "condition"
MESP2_THRESHOLD_SCOPE = "global"
HOST_THRESHOLD_SCALE = 0.85
DONOR_THRESHOLD_SCALE = 0.85
FOXF1_THRESHOLD_SCALE = 0.85
MESP2_THRESHOLD_SCALE = 1.0
FOXF1_THRESHOLD_CONDITION_SCALES = {}
FOXF1_THRESHOLD_CONDITION_VALUES = {}
FOXF1_THRESHOLD_FILE_SCALES = {
    # Example: 4: 1.15 would raise the FOXF1 threshold for file_id 4 by 15%.
}
FOXF1_COMPONENT_RADIUS_PX = 80.0
FOXF1_HOST_SEED_THRESHOLD_SCALE = 1.15
NORMALIZATION_SAMPLE_PER_PLANE = 50000

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("RERUN_PIPELINE:", RERUN_PIPELINE)
print("NORMALIZATION_OUTPUT:", NORMALIZATION_OUTPUT)
print("NORMALIZATION_HIST_OUTPUT:", NORMALIZATION_HIST_OUTPUT)
print("THRESHOLD_REFERENCE_CONDITION:", THRESHOLD_REFERENCE_CONDITION)
print("HOST_THRESHOLD_SCOPE:", HOST_THRESHOLD_SCOPE)
print("DONOR_THRESHOLD_SCOPE:", DONOR_THRESHOLD_SCOPE)
print("FOXF1_THRESHOLD_SCOPE:", FOXF1_THRESHOLD_SCOPE)
print("HOST_THRESHOLD_SCALE:", HOST_THRESHOLD_SCALE)
print("DONOR_THRESHOLD_SCALE:", DONOR_THRESHOLD_SCALE)
print("FOXF1_THRESHOLD_SCALE:", FOXF1_THRESHOLD_SCALE)
print("FOXF1_THRESHOLD_CONDITION_SCALES:", FOXF1_THRESHOLD_CONDITION_SCALES)
print("FOXF1_THRESHOLD_CONDITION_VALUES:", FOXF1_THRESHOLD_CONDITION_VALUES)
print("FOXF1_THRESHOLD_FILE_SCALES:", FOXF1_THRESHOLD_FILE_SCALES)
print("FOXF1_COMPONENT_RADIUS_PX:", FOXF1_COMPONENT_RADIUS_PX)
print("FOXF1_HOST_SEED_THRESHOLD_SCALE:", FOXF1_HOST_SEED_THRESHOLD_SCALE)
print("NORMALIZATION_SAMPLE_PER_PLANE:", NORMALIZATION_SAMPLE_PER_PLANE)
print("HOST_FOXF1_BAR_PNG:", HOST_FOXF1_BAR_PNG)
print("HOST_FOXF1_BAR_PDF:", HOST_FOXF1_BAR_PDF)

if RERUN_PIPELINE:
    quant_res = rpq.run_quantification_pipeline(
        root=ROOT,
        data_dir=DATA_DIR,
        foxf1_identity_output=FOXF1_IDENTITY_OUTPUT,
        normalization_output=NORMALIZATION_OUTPUT,
        normalization_hist_output=NORMALIZATION_HIST_OUTPUT,
        threshold_method="log1p_otsu",
        normalization_sample_per_plane=NORMALIZATION_SAMPLE_PER_PLANE,
        threshold_reference_condition=THRESHOLD_REFERENCE_CONDITION,
        host_threshold_scope=HOST_THRESHOLD_SCOPE,
        donor_threshold_scope=DONOR_THRESHOLD_SCOPE,
        foxf1_threshold_scope=FOXF1_THRESHOLD_SCOPE,
        mesp2_threshold_scope=MESP2_THRESHOLD_SCOPE,
        host_threshold_scale=HOST_THRESHOLD_SCALE,
        donor_threshold_scale=DONOR_THRESHOLD_SCALE,
        foxf1_threshold_scale=FOXF1_THRESHOLD_SCALE,
        mesp2_threshold_scale=MESP2_THRESHOLD_SCALE,
        foxf1_threshold_condition_scales=FOXF1_THRESHOLD_CONDITION_SCALES,
        foxf1_threshold_condition_values=FOXF1_THRESHOLD_CONDITION_VALUES,
        foxf1_threshold_file_scales=FOXF1_THRESHOLD_FILE_SCALES,
        foxf1_component_radius_px=FOXF1_COMPONENT_RADIUS_PX,
        foxf1_host_seed_threshold_scale=FOXF1_HOST_SEED_THRESHOLD_SCALE,
        write_outputs=True,
    )
    normalization_df = quant_res["normalization_df"].copy()
    normalization_hist_df = quant_res["normalization_hist_df"].copy()
    threshold_df = quant_res["threshold_df"].copy()
    plane_metrics_df = quant_res["plane_metrics_df"].copy()
    class_summary_df = quant_res["class_summary_df"].copy()
    foxf1_identity_df = quant_res["foxf1_identity_df"].copy()
    sampled_pixels_df = quant_res["sampled_pixels_df"].copy()
else:
    normalization_df = pd.read_csv(NORMALIZATION_OUTPUT, sep="\t")
    normalization_hist_df = pd.read_csv(NORMALIZATION_HIST_OUTPUT, sep="\t")
    threshold_df = pd.read_csv(THRESHOLD_OUTPUT, sep="\t")
    plane_metrics_df = pd.read_csv(PLANE_METRICS_OUTPUT, sep="\t")
    class_summary_df = pd.read_csv(CLASS_SUMMARY_OUTPUT, sep="\t")
    foxf1_identity_df = pd.read_csv(FOXF1_IDENTITY_OUTPUT, sep="\t")
    sampled_pixels_df = pd.read_csv(PIXEL_SAMPLE_OUTPUT, sep="\t")

display(normalization_df)
display(threshold_df)

            ## Condition-Level Off-Mask Null Fit Review

            These histograms show the pooled raw off-mask pixels for each image, pooled across all z planes.
            Within each condition and channel, the three image-level histograms are drawn on the same axes so the raw intensity x-values can be compared directly before any normalization or DAPI ratioing.
            

In [ ]:
CHANNEL_ORDER = ["dapi", "host", "donor", "foxf1", "mesp2"]
CHANNEL_LABELS = {
    "dapi": "DAPI",
    "host": "Host reporter",
    "donor": "Donor reporter",
    "foxf1": "FOXF1",
    "mesp2": "MESP2",
}
HIST_BINS_PER_CONDITION_CHANNEL = 48
OFFMASK_SAMPLE_PER_Z = 50000
RNG = np.random.default_rng(7)
POSITION_COLORS = ["#1e88e5", "#d81b60", "#00897b", "#ef6c00", "#6d4c41"]


def collect_offmask_raw_by_file(channel_key: str) -> dict[int, np.ndarray]:
    file_meta = (
        plane_metrics_df[["condition", "file_id", "position_label", "file_name", "file_path"]]
        .drop_duplicates()
        .sort_values(["condition", "file_id"])
        .reset_index(drop=True)
    )
    pooled: dict[int, np.ndarray] = {}
    for meta in file_meta.itertuples(index=False):
        stack = tqh.load_transplant_stack(ROOT / str(meta.file_path))
        channel_idx = tqh.channel_index(stack.canonical_channel_names, channel_key)
        file_rows = plane_metrics_df[plane_metrics_df["file_id"] == int(meta.file_id)].sort_values("z_index")
        chunks: list[np.ndarray] = []
        for row in file_rows.itertuples(index=False):
            raw = np.asarray(stack.data_czyx[channel_idx, int(row.z_index)], dtype=np.float32)
            mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
            offmask = ~mask
            sampled = tqh.sample_masked_values(
                values=raw,
                mask=offmask,
                sample_size=int(OFFMASK_SAMPLE_PER_Z),
                rng=RNG,
            )
            if sampled.size:
                chunks.append(sampled.astype(np.float32))
        pooled[int(meta.file_id)] = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros((0,), dtype=np.float32)
    return pooled


offmask_raw_by_channel = {channel_key: collect_offmask_raw_by_file(channel_key) for channel_key in CHANNEL_ORDER}
file_meta = (
    plane_metrics_df[["condition", "file_id", "position_label", "file_name"]]
    .drop_duplicates()
    .sort_values(["condition", "file_id"])
    .reset_index(drop=True)
)
condition_order = sorted(
    file_meta["condition"].astype(str).dropna().unique().tolist(),
    key=lambda cond: (0 if str(cond).lower() == "ctrl" else 1, str(cond).lower()),
)

def compute_channel_edges(values_by_file: dict[int, np.ndarray]) -> np.ndarray | None:
    channel_chunks = [
        np.asarray(values, dtype=np.float32)
        for values in values_by_file.values()
        if np.asarray(values).size
    ]
    if not channel_chunks:
        return None
    pooled_channel = np.concatenate(channel_chunks).astype(np.float32)
    pooled_channel = pooled_channel[np.isfinite(pooled_channel)]
    if pooled_channel.size == 0:
        return None
    lo = float(np.quantile(pooled_channel, 0.001))
    hi = float(np.quantile(pooled_channel, 0.999))
    if not np.isfinite(lo):
        lo = float(np.min(pooled_channel))
    if not np.isfinite(hi):
        hi = float(np.max(pooled_channel))
    if hi <= lo:
        hi = lo + 1.0
    pad = 0.05 * (hi - lo)
    return np.linspace(
        lo - pad,
        hi + pad,
        HIST_BINS_PER_CONDITION_CHANNEL + 1,
        dtype=np.float64,
    )


def plot_offmask_histograms(
    *,
    values_by_channel: dict[str, dict[int, np.ndarray]],
    x_label: str,
    figure_title_suffix: str,
) -> None:
    channel_edges = {
        channel_key: compute_channel_edges(values_by_channel[channel_key])
        for channel_key in CHANNEL_ORDER
    }

    for condition in condition_order:
        cond_meta = file_meta[file_meta["condition"].astype(str) == str(condition)].copy().sort_values("file_id")
        if cond_meta.empty:
            continue
        fig, axes = plt.subplots(1, len(CHANNEL_ORDER), figsize=(4.2 * len(CHANNEL_ORDER), 3.6), sharey=False)
        axes = np.asarray(axes).reshape(-1)

        for col_idx, channel_key in enumerate(CHANNEL_ORDER):
            ax = axes[col_idx]
            value_list: list[np.ndarray] = []
            labels: list[str] = []
            for meta in cond_meta.itertuples(index=False):
                values = np.asarray(
                    values_by_channel[channel_key].get(int(meta.file_id), np.zeros((0,), dtype=np.float32)),
                    dtype=np.float32,
                )
                values = values[np.isfinite(values)]
                if values.size == 0:
                    continue
                value_list.append(values)
                labels.append(str(meta.position_label))
            if not value_list:
                ax.axis("off")
                continue

            edges = channel_edges.get(channel_key)
            if edges is None:
                ax.axis("off")
                continue

            for idx, (values, label) in enumerate(zip(value_list, labels)):
                color = POSITION_COLORS[idx % len(POSITION_COLORS)]
                clipped = np.clip(values.astype(np.float64), edges[0], edges[-1])
                ax.hist(
                    clipped,
                    bins=edges,
                    histtype="step",
                    density=True,
                    linewidth=1.6,
                    color=color,
                    label=label,
                )

            ax.set_title(f"{condition} | {CHANNEL_LABELS[channel_key]}")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Normalized density")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.legend(frameon=False, fontsize=8)

        fig.suptitle(f"{condition} {figure_title_suffix}", y=1.02, fontsize=16)
        plt.tight_layout()
        plt.show()


plot_offmask_histograms(
    values_by_channel=offmask_raw_by_channel,
    x_label="Raw off-mask intensity",
    figure_title_suffix="off-mask raw histograms by position (pooled across z)",
)

            ## Off-Mask Histograms After Per-Image Median Subtraction

            These plots use the exact same pooled off-mask pixels as above, but each image is recentered by subtracting its own pooled off-mask median for that channel. This is just a visualization/debug step so we can judge whether per-image background subtraction makes the distributions align more cleanly before any DAPI ratioing.
            

In [ ]:
offmask_median_subtracted_by_channel: dict[str, dict[int, np.ndarray]] = {}
offmask_median_table_rows: list[dict[str, object]] = []

for channel_key in CHANNEL_ORDER:
    centered_by_file: dict[int, np.ndarray] = {}
    for meta in file_meta.itertuples(index=False):
        values = np.asarray(
            offmask_raw_by_channel[channel_key].get(int(meta.file_id), np.zeros((0,), dtype=np.float32)),
            dtype=np.float32,
        )
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            centered_by_file[int(meta.file_id)] = np.zeros((0,), dtype=np.float32)
            continue
        median_value = float(np.median(finite))
        centered_values = (finite.astype(np.float32) - np.float32(median_value)).astype(np.float32)
        centered_by_file[int(meta.file_id)] = centered_values
        offmask_median_table_rows.append(
            {
                "condition": meta.condition,
                "file_id": int(meta.file_id),
                "position_label": meta.position_label,
                "channel_key": channel_key,
                "offmask_median_raw": median_value,
                "n_offmask_pixels_sampled": int(finite.size),
            }
        )
    offmask_median_subtracted_by_channel[channel_key] = centered_by_file

offmask_median_table = pd.DataFrame(offmask_median_table_rows).sort_values(
    ["channel_key", "condition", "file_id"]
).reset_index(drop=True)
display(offmask_median_table)

plot_offmask_histograms(
    values_by_channel=offmask_median_subtracted_by_channel,
    x_label="Raw off-mask intensity - per-image off-mask median",
    figure_title_suffix="off-mask histograms after per-image median subtraction",
)

            ## On-Mask Histograms By Image After Per-Image Off-Mask Median Subtraction

            This is the image-level on-mask companion to the section above: each row is one condition, and each panel overlays the three image-level on-mask histograms for that condition on shared channel axes.
            

In [ ]:
def collect_onmask_raw_by_file(channel_key: str) -> dict[int, np.ndarray]:
    pooled: dict[int, np.ndarray] = {}
    file_meta_local = (
        plane_metrics_df[["condition", "file_id", "position_label", "file_name", "file_path"]]
        .drop_duplicates()
        .sort_values(["condition", "file_id"])
        .reset_index(drop=True)
    )
    for meta in file_meta_local.itertuples(index=False):
        stack = tqh.load_transplant_stack(ROOT / str(meta.file_path))
        channel_idx = tqh.channel_index(stack.canonical_channel_names, channel_key)
        file_rows = plane_metrics_df[plane_metrics_df["file_id"] == int(meta.file_id)].sort_values("z_index")
        chunks: list[np.ndarray] = []
        for row in file_rows.itertuples(index=False):
            raw = np.asarray(stack.data_czyx[channel_idx, int(row.z_index)], dtype=np.float32)
            mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
            sampled = tqh.sample_masked_values(
                values=raw,
                mask=mask,
                sample_size=int(OFFMASK_SAMPLE_PER_Z),
                rng=RNG,
            )
            if sampled.size:
                chunks.append(sampled.astype(np.float32))
        pooled[int(meta.file_id)] = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros((0,), dtype=np.float32)
    return pooled


offmask_median_lookup = {
    (str(row.channel_key), int(row.file_id)): float(row.offmask_median_raw)
    for row in offmask_median_table.itertuples(index=False)
}

onmask_raw_by_channel = {
    channel_key: collect_onmask_raw_by_file(channel_key)
    for channel_key in CHANNEL_ORDER
}

onmask_centered_by_channel: dict[str, dict[int, np.ndarray]] = {}
for channel_key in CHANNEL_ORDER:
    centered_by_file: dict[int, np.ndarray] = {}
    for file_id, values in onmask_raw_by_channel[channel_key].items():
        values = np.asarray(values, dtype=np.float32)
        values = values[np.isfinite(values)]
        if values.size == 0:
            centered_by_file[int(file_id)] = np.zeros((0,), dtype=np.float32)
            continue
        background_median = float(offmask_median_lookup[(str(channel_key), int(file_id))])
        centered_by_file[int(file_id)] = (values - np.float32(background_median)).astype(np.float32)
    onmask_centered_by_channel[channel_key] = centered_by_file

onmask_channel_edges = {
    channel_key: compute_channel_edges(onmask_centered_by_channel[channel_key])
    for channel_key in CHANNEL_ORDER
}

fig, axes = plt.subplots(len(condition_order), len(CHANNEL_ORDER), figsize=(4.2 * len(CHANNEL_ORDER), 3.6 * len(condition_order)), sharey=False)
axes = np.asarray(axes)
if axes.ndim == 1:
    axes = axes.reshape(1, -1)

for row_idx, condition in enumerate(condition_order):
    cond_meta = file_meta[file_meta["condition"].astype(str) == str(condition)].copy().sort_values("file_id")
    for col_idx, channel_key in enumerate(CHANNEL_ORDER):
        ax = axes[row_idx, col_idx]
        edges = onmask_channel_edges.get(channel_key)
        if edges is None:
            ax.axis("off")
            continue

        drew_any = False
        for idx, meta in enumerate(cond_meta.itertuples(index=False)):
            values = np.asarray(
                onmask_centered_by_channel[channel_key].get(int(meta.file_id), np.zeros((0,), dtype=np.float32)),
                dtype=np.float32,
            )
            values = values[np.isfinite(values)]
            if values.size == 0:
                continue
            drew_any = True
            color = POSITION_COLORS[idx % len(POSITION_COLORS)]
            clipped = np.clip(values.astype(np.float64), edges[0], edges[-1])
            ax.hist(
                clipped,
                bins=edges,
                histtype="step",
                density=True,
                linewidth=1.6,
                color=color,
                label=str(meta.position_label),
            )

        if not drew_any:
            ax.axis("off")
            continue

        ax.set_title(f"{condition} | {CHANNEL_LABELS[channel_key]}")
        ax.set_xlabel("Raw on-mask intensity - per-image off-mask median")
        ax.set_ylabel("Normalized density")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(frameon=False, fontsize=8)

fig.suptitle("On-mask histograms by image after per-image off-mask median subtraction", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

            ## Condition-Pooled Off-Mask Histograms After Per-Image Median Subtraction

            These plots pool the per-image-median-subtracted off-mask pixels within each condition, then overlay the three conditions on the same axes for each channel. This is meant to show whether background-subtracted null distributions are similar enough to compare at the condition level.
            

In [ ]:
CONDITION_COLORS = {
    "ctrl": "#1e88e5",
    "BMPR1A_KD": "#d81b60",
    "LDN": "#00897b",
}

condition_pooled_centered_by_channel: dict[str, dict[str, np.ndarray]] = {}
for channel_key in CHANNEL_ORDER:
    pooled_by_condition: dict[str, np.ndarray] = {}
    for condition in condition_order:
        cond_file_ids = (
            file_meta[file_meta["condition"].astype(str) == str(condition)]["file_id"]
            .astype(int)
            .tolist()
        )
        chunks = [
            np.asarray(offmask_median_subtracted_by_channel[channel_key].get(file_id, np.zeros((0,), dtype=np.float32)), dtype=np.float32)
            for file_id in cond_file_ids
        ]
        chunks = [chunk[np.isfinite(chunk)] for chunk in chunks if chunk.size]
        pooled_by_condition[str(condition)] = (
            np.concatenate(chunks).astype(np.float32) if chunks else np.zeros((0,), dtype=np.float32)
        )
    condition_pooled_centered_by_channel[channel_key] = pooled_by_condition

condition_channel_edges = {
    channel_key: compute_channel_edges(
        {
            idx: values
            for idx, values in enumerate(condition_pooled_centered_by_channel[channel_key].values())
        }
    )
    for channel_key in CHANNEL_ORDER
}

fig, axes = plt.subplots(1, len(CHANNEL_ORDER), figsize=(4.4 * len(CHANNEL_ORDER), 3.8), sharey=False)
axes = np.asarray(axes).reshape(-1)

for col_idx, channel_key in enumerate(CHANNEL_ORDER):
    ax = axes[col_idx]
    edges = condition_channel_edges.get(channel_key)
    if edges is None:
        ax.axis("off")
        continue

    drew_any = False
    for condition in condition_order:
        values = np.asarray(
            condition_pooled_centered_by_channel[channel_key].get(str(condition), np.zeros((0,), dtype=np.float32)),
            dtype=np.float32,
        )
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        drew_any = True
        color = CONDITION_COLORS.get(str(condition), None)
        clipped = np.clip(values.astype(np.float64), edges[0], edges[-1])
        ax.hist(
            clipped,
            bins=edges,
            histtype="step",
            density=True,
            linewidth=1.8,
            color=color,
            label=str(condition),
        )

    if not drew_any:
        ax.axis("off")
        continue

    ax.set_title(CHANNEL_LABELS[channel_key])
    ax.set_xlabel("Raw off-mask intensity - per-image off-mask median")
    ax.set_ylabel("Normalized density")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Condition-pooled off-mask histograms after per-image median subtraction", y=1.03, fontsize=16)
plt.tight_layout()
plt.show()

            ## Condition-Pooled On-Mask Histograms After Per-Image Off-Mask Median Subtraction

            These plots use the on-mask pixels from each image, but still subtract that image's own off-mask median for the same channel before pooling within condition. This makes the tissue-pixel distributions directly comparable to the background-centered off-mask plots above.
            

In [ ]:
def collect_onmask_raw_by_file(channel_key: str) -> dict[int, np.ndarray]:
    pooled: dict[int, np.ndarray] = {}
    file_meta_local = (
        plane_metrics_df[["condition", "file_id", "position_label", "file_name", "file_path"]]
        .drop_duplicates()
        .sort_values(["condition", "file_id"])
        .reset_index(drop=True)
    )
    for meta in file_meta_local.itertuples(index=False):
        stack = tqh.load_transplant_stack(ROOT / str(meta.file_path))
        channel_idx = tqh.channel_index(stack.canonical_channel_names, channel_key)
        file_rows = plane_metrics_df[plane_metrics_df["file_id"] == int(meta.file_id)].sort_values("z_index")
        chunks: list[np.ndarray] = []
        for row in file_rows.itertuples(index=False):
            raw = np.asarray(stack.data_czyx[channel_idx, int(row.z_index)], dtype=np.float32)
            mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
            sampled = tqh.sample_masked_values(
                values=raw,
                mask=mask,
                sample_size=int(OFFMASK_SAMPLE_PER_Z),
                rng=RNG,
            )
            if sampled.size:
                chunks.append(sampled.astype(np.float32))
        pooled[int(meta.file_id)] = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros((0,), dtype=np.float32)
    return pooled


offmask_median_lookup = {
    (str(row.channel_key), int(row.file_id)): float(row.offmask_median_raw)
    for row in offmask_median_table.itertuples(index=False)
}

onmask_raw_by_channel = {
    channel_key: collect_onmask_raw_by_file(channel_key)
    for channel_key in CHANNEL_ORDER
}

onmask_centered_by_channel: dict[str, dict[int, np.ndarray]] = {}
for channel_key in CHANNEL_ORDER:
    centered_by_file: dict[int, np.ndarray] = {}
    for file_id, values in onmask_raw_by_channel[channel_key].items():
        values = np.asarray(values, dtype=np.float32)
        values = values[np.isfinite(values)]
        if values.size == 0:
            centered_by_file[int(file_id)] = np.zeros((0,), dtype=np.float32)
            continue
        background_median = float(offmask_median_lookup[(str(channel_key), int(file_id))])
        centered_by_file[int(file_id)] = (values - np.float32(background_median)).astype(np.float32)
    onmask_centered_by_channel[channel_key] = centered_by_file

condition_pooled_onmask_centered_by_channel: dict[str, dict[str, np.ndarray]] = {}
for channel_key in CHANNEL_ORDER:
    pooled_by_condition: dict[str, np.ndarray] = {}
    for condition in condition_order:
        cond_file_ids = (
            file_meta[file_meta["condition"].astype(str) == str(condition)]["file_id"]
            .astype(int)
            .tolist()
        )
        chunks: list[np.ndarray] = []
        for file_id in cond_file_ids:
            values = np.asarray(
                onmask_centered_by_channel[channel_key].get(file_id, np.zeros((0,), dtype=np.float32)),
                dtype=np.float32,
            )
            values = values[np.isfinite(values)]
            if values.size == 0:
                continue
            chunks.append(values.astype(np.float32))
        pooled_by_condition[str(condition)] = (
            np.concatenate(chunks).astype(np.float32) if chunks else np.zeros((0,), dtype=np.float32)
        )
    condition_pooled_onmask_centered_by_channel[channel_key] = pooled_by_condition

condition_onmask_edges = {
    channel_key: compute_channel_edges(
        {
            idx: values
            for idx, values in enumerate(condition_pooled_onmask_centered_by_channel[channel_key].values())
        }
    )
    for channel_key in CHANNEL_ORDER
}

fig, axes = plt.subplots(1, len(CHANNEL_ORDER), figsize=(4.4 * len(CHANNEL_ORDER), 3.8), sharey=False)
axes = np.asarray(axes).reshape(-1)

for col_idx, channel_key in enumerate(CHANNEL_ORDER):
    ax = axes[col_idx]
    edges = condition_onmask_edges.get(channel_key)
    if edges is None:
        ax.axis("off")
        continue

    drew_any = False
    for condition in condition_order:
        values = np.asarray(
            condition_pooled_onmask_centered_by_channel[channel_key].get(str(condition), np.zeros((0,), dtype=np.float32)),
            dtype=np.float32,
        )
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        drew_any = True
        color = CONDITION_COLORS.get(str(condition), None)
        clipped = np.clip(values.astype(np.float64), edges[0], edges[-1])
        ax.hist(
            clipped,
            bins=edges,
            histtype="step",
            density=True,
            linewidth=1.8,
            color=color,
            label=str(condition),
        )

    if not drew_any:
        ax.axis("off")
        continue

    ax.set_title(CHANNEL_LABELS[channel_key])
    ax.set_xlabel("Raw on-mask intensity - per-image off-mask median")
    ax.set_ylabel("Normalized density")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Condition-pooled on-mask histograms after per-image off-mask median subtraction", y=1.03, fontsize=16)
plt.tight_layout()
plt.show()

            ## Threshold Debug Views

            These plots show where the current ratio thresholds fall relative to the sampled masked-pixel distributions.
            Host, donor, and FOXF1 thresholds now vary by condition, while MESP2 remains global.
            The displayed FOXF1-positive region is the donor-adjacent connected component of the thresholded FOXF1 mask, matching the current quantification rule.
            

In [ ]:
def get_threshold(channel_key: str, condition: str | None = None, file_id: int | None = None) -> float:
    channel_rows = threshold_df[threshold_df["channel_key"] == str(channel_key)].copy()
    if channel_rows.empty:
        raise KeyError(f"No threshold rows found for channel {channel_key!r}")
    scope = str(channel_rows["threshold_scope"].iloc[0])
    if scope == "global":
        return float(channel_rows["threshold_value"].iloc[0])
    if scope == "condition":
        sub = channel_rows[channel_rows["threshold_group_condition"].astype(str) == str(condition)]
        if sub.empty:
            raise KeyError(f"No condition threshold for channel {channel_key!r} and condition {condition!r}")
        return float(sub["threshold_value"].iloc[0])
    if scope == "file":
        file_ids = pd.to_numeric(channel_rows["threshold_group_file_id"], errors="coerce")
        sub = channel_rows[file_ids == int(file_id)]
        if sub.empty:
            raise KeyError(f"No file threshold for channel {channel_key!r} and file_id {file_id!r}")
        return float(sub["threshold_value"].iloc[0])
    raise ValueError(f"Unsupported threshold scope {scope!r} for channel {channel_key!r}")


def subtract_offmask_background(signal: np.ndarray, organoid_mask: np.ndarray) -> np.ndarray:
    corrected, _, _ = tqh.background_correct_signal(
        signal=np.asarray(signal, dtype=np.float32),
        organoid_mask=np.asarray(organoid_mask, dtype=bool),
        background_estimator="whole_off_organoid",
    )
    return corrected


def compute_host_seed(host_ratio: np.ndarray, organoid_mask: np.ndarray, donor_graft: np.ndarray, host_threshold: float) -> tuple[np.ndarray, float]:
    seed_threshold = float(host_threshold) * float(FOXF1_HOST_SEED_THRESHOLD_SCALE)
    seed_raw = (
        np.asarray(organoid_mask, dtype=bool)
        & np.isfinite(np.asarray(host_ratio, dtype=np.float32))
        & (np.asarray(host_ratio, dtype=np.float32) > seed_threshold)
        & (~np.asarray(donor_graft, dtype=bool))
    )
    seed = tqh.largest_connected_component(seed_raw)
    if np.any(seed):
        seed = ndi.binary_fill_holes(seed) & np.asarray(organoid_mask, dtype=bool) & (~np.asarray(donor_graft, dtype=bool))
    else:
        host_score = np.asarray(host_ratio, dtype=np.float32) / max(float(host_threshold), 1e-6)
        available = np.asarray(organoid_mask, dtype=bool) & (~np.asarray(donor_graft, dtype=bool)) & np.isfinite(host_score)
        if np.any(available):
            seed = np.zeros_like(np.asarray(organoid_mask, dtype=bool), dtype=bool)
            seed.flat[int(np.nanargmax(np.where(available, host_score, -np.inf)))] = True
    return np.asarray(seed, dtype=bool), float(seed_threshold)


def compute_lineage_partition(
    organoid_mask: np.ndarray,
    donor_graft: np.ndarray,
    host_seed: np.ndarray,
    donor_ratio: np.ndarray,
    host_ratio: np.ndarray,
    donor_threshold: float,
    host_threshold: float,
) -> tuple[np.ndarray, np.ndarray]:
    donor_score = np.asarray(donor_ratio, dtype=np.float32) / max(float(donor_threshold), 1e-6)
    host_score = np.asarray(host_ratio, dtype=np.float32) / max(float(host_threshold), 1e-6)
    donor_part, host_part = tqh.partition_mask_by_intensity_with_seed_components(
        mask=np.asarray(organoid_mask, dtype=bool),
        primary_seed=np.asarray(donor_graft, dtype=bool),
        secondary_seed=np.asarray(host_seed, dtype=bool),
        primary_score=donor_score,
        secondary_score=host_score,
    )
    return np.asarray(donor_part, dtype=bool), np.asarray(host_part, dtype=bool)


CHANNEL_TITLE = {
    "host": "Host Reporter / DAPI",
    "donor": "Donor Reporter / DAPI",
    "foxf1": "FOXF1 / DAPI",
    "mesp2": "MESP2 / DAPI",
}
CHANNEL_VALUE_COL = {
    "host": "host_ratio",
    "donor": "donor_ratio",
    "foxf1": "foxf1_ratio",
    "mesp2": "mesp2_ratio",
}
CONDITION_COLOR = {
    "ctrl": "#424242",
    "BMPR1A_KD": "#ef6c00",
    "LDN": "#00897b",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()
for ax, channel_key in zip(axes, ["host", "donor", "foxf1", "mesp2"]):
    value_col = CHANNEL_VALUE_COL[channel_key]
    values = sampled_pixels_df[value_col].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    values = np.clip(values, a_min=0.0, a_max=None)
    values_log = np.log1p(values)
    ax.hist(values_log, bins=120, color="#b0bec5", edgecolor="none")
    channel_rows = threshold_df[threshold_df["channel_key"] == channel_key].copy()
    scope = str(channel_rows["threshold_scope"].iloc[0]) if not channel_rows.empty else "unknown"
    if scope == "global":
        thr = float(channel_rows["threshold_value"].iloc[0])
        ax.axvline(np.log1p(thr), color="#d32f2f", linewidth=1.8, linestyle="--")
        threshold_label = f"global={thr:.4f}"
    else:
        threshold_label_parts = []
        for thr_row in channel_rows.itertuples(index=False):
            condition = str(getattr(thr_row, "threshold_group_condition", "") or "")
            color = CONDITION_COLOR.get(condition, "#6d4c41")
            thr = float(thr_row.threshold_value)
            ax.axvline(np.log1p(thr), color=color, linewidth=1.2, linestyle="--", alpha=0.7)
            if scope == "condition":
                threshold_label_parts.append(f"{condition}={thr:.4f}")
        if scope == "condition":
            threshold_label = " | ".join(threshold_label_parts)
        elif scope == "file":
            cond_summary = (
                channel_rows.groupby("threshold_group_condition")["threshold_value"]
                .agg(["min", "max", "count"])
                .reset_index()
            )
            threshold_label = " | ".join(
                f"{r.threshold_group_condition}:{r['min']:.4f}-{r['max']:.4f} (n={int(r['count'])})"
                for _, r in cond_summary.iterrows()
            )
        else:
            threshold_label = scope
    ax.set_title(f"{CHANNEL_TITLE[channel_key]}\n{scope} thresholds: {threshold_label}")
    ax.set_xlabel(f"log1p({value_col})")
    ax.set_ylabel("sampled masked pixels")

plt.tight_layout()
plt.show()

            ## FOXF1 Distribution By Image

            These plots show the sampled masked-pixel `FOXF1 / DAPI` distributions for each image separately, with the active FOXF1 threshold drawn for that image.
            

In [ ]:
foxf1_distribution_summary = (
    sampled_pixels_df.groupby(["condition", "file_id", "position_label", "file_name"], as_index=False)
    .agg(
        n_sampled_pixels=("foxf1_ratio", "size"),
        foxf1_ratio_median=("foxf1_ratio", "median"),
        foxf1_ratio_p90=("foxf1_ratio", lambda s: float(np.nanpercentile(pd.to_numeric(s, errors="coerce"), 90))),
        foxf1_positive_fraction=("foxf1_positive", "mean"),
    )
    .sort_values(["condition", "file_id"])
    .reset_index(drop=True)
)
display(foxf1_distribution_summary)


def plot_foxf1_distributions_by_file() -> None:
    plot_df = sampled_pixels_df.copy()
    plot_df["foxf1_ratio"] = pd.to_numeric(plot_df["foxf1_ratio"], errors="coerce")
    plot_df = plot_df[np.isfinite(plot_df["foxf1_ratio"])].copy()
    if plot_df.empty:
        print("No FOXF1 sampled-pixel values available.")
        return

    file_meta = (
        plot_df[["condition", "file_id", "position_label"]]
        .drop_duplicates()
        .sort_values(["condition", "file_id"])
        .reset_index(drop=True)
    )
    n_files = len(file_meta)
    n_cols = 3
    n_rows = int(np.ceil(n_files / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.3 * n_cols, 3.5 * n_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(-1)

    max_log_x = float(np.nanquantile(np.log1p(np.clip(plot_df["foxf1_ratio"].to_numpy(dtype=float), a_min=0.0, a_max=None)), 0.995))
    if not np.isfinite(max_log_x) or max_log_x <= 0:
        max_log_x = 1.0

    for ax, meta in zip(axes, file_meta.itertuples(index=False)):
        sub = plot_df[plot_df["file_id"] == int(meta.file_id)].copy()
        values = np.clip(sub["foxf1_ratio"].to_numpy(dtype=float), a_min=0.0, a_max=None)
        values_log = np.log1p(values)
        color = CONDITION_COLOR.get(str(meta.condition), "#546e7a")
        threshold = float(get_threshold("foxf1", condition=str(meta.condition), file_id=int(meta.file_id)))
        positive_fraction = float(np.nanmean(sub["foxf1_positive"].astype(float))) if len(sub) else np.nan

        ax.hist(values_log, bins=80, density=True, color=color, alpha=0.75, edgecolor="none")
        ax.axvline(np.log1p(threshold), color="#d32f2f", linewidth=1.5, linestyle="--")
        ax.set_title(
            f"{meta.condition} | {meta.position_label}\n"
            f"thr={threshold:.3f} | FOXF1+={100.0 * positive_fraction:.1f}%"
        )
        ax.set_xlim(0.0, max_log_x * 1.02)
        ax.set_xlabel("log1p(FOXF1 / DAPI)")
        ax.set_ylabel("Normalized sampled-pixel density")

    for ax in axes[n_files:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_foxf1_distributions_by_file()

In [ ]:
class_balance_overall = (
    class_summary_df.groupby("pixel_class_label", as_index=False)
    .agg(
        pixel_count=("pixel_count", "sum"),
        mean_fraction_of_mask=("fraction_of_mask", "mean"),
        mean_foxf1_positive_fraction=("foxf1_positive_fraction", "mean"),
        mean_foxf1_ratio=("foxf1_ratio_mean", "mean"),
    )
    .sort_values("pixel_count", ascending=False)
)
display(class_balance_overall)

In [ ]:
class_balance_by_file = (
    class_summary_df.groupby(["condition", "file_id", "file_name", "pixel_class_label"], as_index=False)
    .agg(
        pixel_count=("pixel_count", "sum"),
        mean_fraction_of_mask=("fraction_of_mask", "mean"),
        mean_foxf1_positive_fraction=("foxf1_positive_fraction", "mean"),
        mean_foxf1_ratio=("foxf1_ratio_mean", "mean"),
    )
    .sort_values(["condition", "file_id", "pixel_class_label"])
)
display(class_balance_by_file)

In [ ]:
class_balance_by_condition = (
    class_summary_df.groupby(["condition", "pixel_class_label"], as_index=False)
    .agg(
        pixel_count=("pixel_count", "sum"),
        mean_fraction_of_mask=("fraction_of_mask", "mean"),
        mean_foxf1_positive_fraction=("foxf1_positive_fraction", "mean"),
        mean_foxf1_ratio=("foxf1_ratio_mean", "mean"),
    )
    .sort_values(["condition", "pixel_class_label"])
)
display(class_balance_by_condition)

In [ ]:
host_donor_plane = (
    class_summary_df[class_summary_df["pixel_class_label"].isin(["host", "donor"])]
    .pivot_table(
        index=["condition", "file_id", "file_name", "z_index"],
        columns="pixel_class_label",
        values=["fraction_of_mask", "foxf1_positive_fraction", "foxf1_ratio_mean"],
    )
    .reset_index()
)
host_donor_plane.columns = [
    "_".join(str(part) for part in col if str(part) != "")
    for col in host_donor_plane.columns.to_flat_index()
]

host_donor_plane["foxf1_ratio_host_minus_donor"] = (
    host_donor_plane["foxf1_ratio_mean_host"] - host_donor_plane["foxf1_ratio_mean_donor"]
)
host_donor_plane["foxf1_positive_host_minus_donor"] = (
    host_donor_plane["foxf1_positive_fraction_host"] - host_donor_plane["foxf1_positive_fraction_donor"]
)

display(host_donor_plane)

            ## FOXF1-Positive Identity Summary

            This is the direct readout we are now using for the bar plot: the entire DAPI-mask is partitioned into a contiguous donor graft region and a contiguous host region using the lineage reporters, and then the donor-adjacent `FOXF1+` domain is intersected with that full lineage map. The older strict host/donor fractions are still shown for reference.

---

**NOTE ADDED FOR THIS REPOSITORY — the published Fig 5n uses the *strict* fractions.**

The sentence above, and the bar plot in the cell below, use `foxf1_positive_host_partition_fraction`.
That reflects where this notebook ended up, not what the paper reports. **Figure 5n plots
`foxf1_positive_host_strict_fraction`**, which is also what the published Source Data column is
headed — *"Host fraction of FOXF1+ (strict)"* — and what `../check_fig5n_stats.py` recomputes.

The two differ little (mean 0.6 percentage points per morph, maximum 1.6), and the effect, its
direction and its significance are the same either way. They differ at all only in how ambiguous
pixels are treated: *strict* counts a pixel only where the lineage reporters identify it
unambiguously, while *partition* assigns every pixel by splitting the whole morph into contiguous
host and donor regions. The ambiguous share is not negligible — in the LDN condition 10.2% of
FOXF1+ pixels are mixed and 21.1% unlabeled — so this is a real methodological choice, and the
published figure takes the conservative one.

The text above is left as it ran, so this cell is an accurate record of the notebook rather than a
rewritten one.


In [ ]:
foxf1_file_summary = (
    foxf1_identity_df[foxf1_identity_df["summary_level"] == "file"]
    .copy()
    .sort_values(["condition", "file_id"])
)

display(
    foxf1_file_summary[
        [
            "condition",
            "file_id",
            "file_name",
            "foxf1_positive_pixels",
            "foxf1_positive_fraction_of_mask",
            "foxf1_positive_host_strict_fraction",
            "foxf1_positive_donor_strict_fraction",
            "foxf1_positive_host_partition_fraction",
            "foxf1_positive_donor_partition_fraction",
            "foxf1_positive_mixed_fraction",
            "foxf1_positive_unlabeled_fraction",
            "host_ratio_mean_in_foxf1_positive",
            "donor_ratio_mean_in_foxf1_positive",
            "foxf1_host_seed_threshold",
            "host_seed_pixels",
        ]
    ]
)

condition_foxf1_summary = (
    foxf1_identity_df[foxf1_identity_df["summary_level"] == "condition"]
    .copy()
    .sort_values("condition")
)
display(condition_foxf1_summary)

global_foxf1_summary = foxf1_identity_df[foxf1_identity_df["summary_level"] == "global"].copy()
display(global_foxf1_summary)

In [ ]:
if not foxf1_file_summary.empty:
    plot_df = foxf1_file_summary[
        ["condition", "file_id", "file_name", "foxf1_positive_host_partition_fraction"]
    ].copy()
    plot_df = plot_df[np.isfinite(plot_df["foxf1_positive_host_partition_fraction"])].reset_index(drop=True)
    plot_df["host_percent_of_foxf1_positive"] = 100.0 * plot_df["foxf1_positive_host_partition_fraction"]

    condition_order = sorted(
        plot_df["condition"].dropna().unique().tolist(),
        key=lambda cond: (0 if str(cond).lower() == "ctrl" else 1, str(cond).lower()),
    )
    plot_df["condition"] = pd.Categorical(plot_df["condition"], categories=condition_order, ordered=True)
    plot_df = plot_df.sort_values(["condition", "file_id"]).reset_index(drop=True)

    summary_df = (
        plot_df.groupby("condition", observed=True, as_index=False)
        .agg(
            mean_host_percent=("host_percent_of_foxf1_positive", "mean"),
            n_files=("file_id", "size"),
            sd_host_percent=("host_percent_of_foxf1_positive", "std"),
        )
    )
    summary_df["sem_host_percent"] = (
        summary_df["sd_host_percent"].fillna(0.0) / np.sqrt(summary_df["n_files"].clip(lower=1))
    )

    ctrl_condition = next((cond for cond in condition_order if str(cond).lower() == "ctrl"), condition_order[0])
    ctrl_values = plot_df.loc[
        plot_df["condition"].astype(str) == str(ctrl_condition),
        "foxf1_positive_host_partition_fraction",
    ].to_numpy(dtype=float)
    dunnett_targets = [cond for cond in condition_order if str(cond) != str(ctrl_condition)]
    dunnett_pvalues: dict[str, float] = {}
    if ctrl_values.size and dunnett_targets:
        dunnett_samples = [
            plot_df.loc[
                plot_df["condition"].astype(str) == str(cond),
                "foxf1_positive_host_partition_fraction",
            ].to_numpy(dtype=float)
            for cond in dunnett_targets
        ]
        dunnett_res = dunnett(*dunnett_samples, control=ctrl_values, alternative="two-sided")
        dunnett_pvalues = {
            str(cond): float(pval)
            for cond, pval in zip(dunnett_targets, np.atleast_1d(dunnett_res.pvalue))
        }

    def p_to_stars(pvalue: float) -> str:
        if not np.isfinite(pvalue):
            return "n.s."
        if pvalue < 1e-4:
            return "****"
        if pvalue < 1e-3:
            return "***"
        if pvalue < 1e-2:
            return "**"
        if pvalue < 5e-2:
            return "*"
        return "n.s."

    x_positions = np.arange(len(summary_df), dtype=float)
    fig_width = max(4.6, 2.2 * len(summary_df) + 1.0)
    fig, ax = plt.subplots(figsize=(fig_width, 5.6))
    ax.bar(
        x_positions,
        summary_df["mean_host_percent"].to_numpy(dtype=float),
        yerr=summary_df["sem_host_percent"].to_numpy(dtype=float),
        width=0.62,
        color="#2e7d32",
        edgecolor="#1b1b1b",
        linewidth=1.0,
        capsize=6,
        zorder=2,
    )

    for x_pos, condition in zip(x_positions, summary_df["condition"].tolist()):
        sub = plot_df.loc[plot_df["condition"] == condition].copy()
        jitter = np.linspace(-0.08, 0.08, len(sub)) if len(sub) > 1 else np.asarray([0.0])
        ax.scatter(
            x_pos + jitter,
            sub["host_percent_of_foxf1_positive"].to_numpy(dtype=float),
            s=42,
            color="#1b1b1b",
            zorder=3,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(summary_df["condition"].astype(str).tolist())
    ax.set_ylabel("% FOXF1+ pixels assigned to host")
    ax.set_title("Condition comparison (full-mask lineage assignment)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    y_max = max(100.0, float(np.nanmax(plot_df["host_percent_of_foxf1_positive"])) + 18.0)
    ax.set_ylim(0, y_max)

    cond_to_x = {
        str(cond): float(x_pos)
        for cond, x_pos in zip(summary_df["condition"].astype(str).tolist(), x_positions)
    }
    cond_to_bar_top = {}
    for row in summary_df.itertuples(index=False):
        cond_to_bar_top[str(row.condition)] = float(row.mean_host_percent) + float(row.sem_host_percent)

    bracket_base = max(
        float(np.nanmax(plot_df["host_percent_of_foxf1_positive"])) + 4.0,
        float(np.nanmax(summary_df["mean_host_percent"] + summary_df["sem_host_percent"])) + 2.5,
    )
    bracket_gap = 5.0
    ctrl_x = cond_to_x[str(ctrl_condition)]
    legend_handles: list[Line2D] = []
    legend_labels: list[str] = []
    for idx, cond in enumerate(dunnett_targets):
        cond_str = str(cond)
        cond_x = cond_to_x[cond_str]
        y = bracket_base + idx * bracket_gap
        h = 1.2
        ax.plot([ctrl_x, ctrl_x, cond_x, cond_x], [y, y + h, y + h, y], color="black", linewidth=1.1, clip_on=False)
        pvalue = float(dunnett_pvalues.get(cond_str, np.nan))
        ax.text(
            (ctrl_x + cond_x) / 2.0,
            y + h + 0.6,
            p_to_stars(pvalue),
            ha="center",
            va="bottom",
            fontsize=12,
        )
        legend_handles.append(Line2D([0], [0], color="black", linewidth=1.1))
        legend_labels.append(f"ctrl vs {cond_str}: p = {pvalue:.2e}")

    if legend_handles:
        ax.legend(
            legend_handles,
            legend_labels,
            title="Dunnett (two-sided)",
            loc="upper right",
            frameon=False,
            fontsize=9,
            title_fontsize=9,
        )

    plt.tight_layout()
    fig.savefig(HOST_FOXF1_BAR_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(HOST_FOXF1_BAR_SVG, bbox_inches="tight")
    fig.savefig(HOST_FOXF1_BAR_PDF, bbox_inches="tight")
    plt.show()

    display(summary_df)
    display(plot_df)

            ## Spatial Classification Debug

            These panels show one representative FOXF1-rich z plane per image with both ratio maps and thresholded overlays, so the current FOXF1-positive region and host/donor identity calls can be sanity-checked against the raw signal context.
            

In [ ]:
def _robust_rescale(image: np.ndarray, q_low: float = 0.01, q_high: float = 0.99) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo = float(np.quantile(finite, q_low))
    hi = float(np.quantile(finite, q_high))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


CLASS_COLORS = {
    0: "#9e9e9e",
    1: "#2e7d32",
    2: "#1565c0",
    3: "#ef6c00",
}
FOXF1_POS_CMAP = ListedColormap(["#d81b60"])
HOST_POS_CMAP = ListedColormap(["#2e7d32"])
DONOR_POS_CMAP = ListedColormap(["#1565c0"])
FOXF1_NEG_CMAP = plt.get_cmap("magma").copy()
FOXF1_NEG_CMAP.set_bad(color="#ffffff", alpha=1.0)


def _class_rgb(class_map: np.ndarray) -> np.ndarray:
    out = np.zeros(class_map.shape + (3,), dtype=np.float32)
    for code, color in CLASS_COLORS.items():
        mask = class_map == int(code)
        if not np.any(mask):
            continue
        rgb = np.asarray(
            [int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)],
            dtype=np.float32,
        ) / 255.0
        out[mask] = rgb
    return out


def _foxf1_negative_composite(
    foxf1_ratio: np.ndarray,
    organoid_mask: np.ndarray,
    foxf1_positive: np.ndarray,
    vmax: float,
) -> np.ndarray:
    ratio = np.asarray(foxf1_ratio, dtype=np.float32)
    mask = np.asarray(organoid_mask, dtype=bool)
    positive = np.asarray(foxf1_positive, dtype=bool)
    negative = mask & (~positive)

    scale = float(vmax) if np.isfinite(vmax) and float(vmax) > 0 else 1.0
    norm = np.clip(ratio / scale, 0.0, 1.0)
    rgba = plt.get_cmap("magma")(norm)

    out = np.ones(mask.shape + (3,), dtype=np.float32)
    out[negative] = rgba[negative, :3]
    out[positive] = np.asarray([216, 27, 96], dtype=np.float32) / 255.0
    return out


def _donor_halo_composite(
    dapi_image: np.ndarray,
    donor_positive: np.ndarray,
    donor_halo: np.ndarray,
) -> np.ndarray:
    dapi = np.asarray(dapi_image, dtype=np.float32)
    donor_positive = np.asarray(donor_positive, dtype=bool)
    donor_halo = np.asarray(donor_halo, dtype=bool)

    base = np.repeat(np.clip(dapi[..., None], 0.0, 1.0), 3, axis=2)
    out = base.copy()

    halo_only = donor_halo & (~donor_positive)
    if np.any(halo_only):
        halo_rgb = np.asarray([0, 188, 212], dtype=np.float32) / 255.0
        out[halo_only] = 0.50 * out[halo_only] + 0.50 * halo_rgb
    if np.any(donor_positive):
        donor_rgb = np.asarray([21, 101, 192], dtype=np.float32) / 255.0
        out[donor_positive] = donor_rgb
    return out


representative_planes = (
    foxf1_identity_df[foxf1_identity_df["summary_level"] == "plane"]
    .sort_values(["foxf1_positive_fraction_of_mask", "foxf1_positive_pixels"], ascending=[False, False])
    .groupby("file_id", as_index=False)
    .head(1)
    [["condition", "file_id", "position_label", "file_name", "z_index", "foxf1_positive_fraction_of_mask"]]
    .merge(
        plane_metrics_df[["condition", "file_id", "z_index", "file_path", "mask_path", "dapi_floor"]],
        on=["condition", "file_id", "z_index"],
        how="left",
    )
    .sort_values(["condition", "file_id", "z_index"])
    .reset_index(drop=True)
)

display(representative_planes)

In [ ]:
def plot_classification_debug(rep_df: pd.DataFrame) -> None:
    if rep_df.empty:
        print("No representative planes available.")
        return

    fig, axes = plt.subplots(len(rep_df), 11, figsize=(44, 4.3 * len(rep_df)))
    if len(rep_df) == 1:
        axes = np.asarray([axes])

    for ax_row, row in zip(axes, rep_df.itertuples(index=False)):
        stack = tqh.load_transplant_stack(ROOT / str(row.file_path))
        dapi_idx = tqh.channel_index(stack.canonical_channel_names, "dapi")
        foxf1_idx = tqh.channel_index(stack.canonical_channel_names, "foxf1")
        donor_idx = tqh.channel_index(stack.canonical_channel_names, "donor")
        host_idx = tqh.channel_index(stack.canonical_channel_names, "host")

        z_index = int(row.z_index)
        dapi_raw = np.asarray(stack.data_czyx[dapi_idx, z_index], dtype=np.float32)
        foxf1_raw = np.asarray(stack.data_czyx[foxf1_idx, z_index], dtype=np.float32)
        donor_raw = np.asarray(stack.data_czyx[donor_idx, z_index], dtype=np.float32)
        host_raw = np.asarray(stack.data_czyx[host_idx, z_index], dtype=np.float32)

        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
        condition_name = str(row.condition)
        dapi_corrected = subtract_offmask_background(dapi_raw, organoid_mask=mask)
        dapi_floor = float(row.dapi_floor)

        foxf1_corr = subtract_offmask_background(foxf1_raw, organoid_mask=mask)
        donor_corr = subtract_offmask_background(donor_raw, organoid_mask=mask)
        host_corr = subtract_offmask_background(host_raw, organoid_mask=mask)

        foxf1_ratio = tqh.reporter_over_dapi(foxf1_corr, dapi_corrected, mask, dapi_floor=dapi_floor)
        donor_ratio = tqh.reporter_over_dapi(donor_corr, dapi_corrected, mask, dapi_floor=dapi_floor)
        host_ratio = tqh.reporter_over_dapi(host_corr, dapi_corrected, mask, dapi_floor=dapi_floor)

        foxf1_threshold = float(get_threshold("foxf1", condition=condition_name, file_id=int(row.file_id)))
        host_threshold = float(get_threshold("host", condition=condition_name, file_id=int(row.file_id)))
        donor_threshold = float(get_threshold("donor", condition=condition_name, file_id=int(row.file_id)))
        host_positive = mask & np.isfinite(host_ratio) & (host_ratio > host_threshold)
        donor_positive_raw = mask & np.isfinite(donor_ratio) & (donor_ratio > donor_threshold)
        donor_positive = tqh.donor_graft_component(
            donor_positive=donor_positive_raw,
            organoid_mask=mask,
            fill_holes=True,
        )
        class_map = tqh.classify_host_donor_from_binary_calls(
            host_positive=host_positive,
            donor_positive=donor_positive,
            host_ratio=host_ratio,
            donor_ratio=donor_ratio,
            dominance_margin_log2=0.75,
            organoid_mask=mask,
        )
        foxf1_positive_raw = mask & np.isfinite(foxf1_ratio) & (foxf1_ratio > foxf1_threshold)
        foxf1_positive, foxf1_donor_halo = tqh.donor_adjacent_foxf1_component(
            foxf1_positive=foxf1_positive_raw,
            donor_positive=donor_positive,
            organoid_mask=mask,
            radius_px=FOXF1_COMPONENT_RADIUS_PX,
        )
        host_seed, host_seed_threshold = compute_host_seed(
            host_ratio=host_ratio,
            organoid_mask=mask,
            donor_graft=donor_positive,
            host_threshold=host_threshold,
        )
        donor_partition_full, host_partition_full = compute_lineage_partition(
            organoid_mask=mask,
            donor_graft=donor_positive,
            host_seed=host_seed,
            donor_ratio=donor_ratio,
            host_ratio=host_ratio,
            donor_threshold=donor_threshold,
            host_threshold=host_threshold,
        )
        foxf1_partition_donor = foxf1_positive & donor_partition_full
        foxf1_partition_host = foxf1_positive & host_partition_full

        dapi_disp = _robust_rescale(dapi_raw)
        foxf1_raw_disp = _robust_rescale(foxf1_raw)
        foxf1_vmax = float(np.nanquantile(foxf1_ratio[mask], 0.995)) if np.any(mask) else 1.0
        if not np.isfinite(foxf1_vmax) or foxf1_vmax <= 0:
            foxf1_vmax = 1.0
        ax_row[0].imshow(dapi_disp, cmap="gray")
        ax_row[0].set_title(f"{row.condition} | {row.position_label} | z{z_index}\nDAPI")
        ax_row[0].axis("off")

        ax_row[1].imshow(foxf1_raw_disp, cmap="magma")
        ax_row[1].set_title("Raw FOXF1\n(contrasted)")
        ax_row[1].axis("off")

        foxf1_disp = np.ma.masked_where(~mask, foxf1_ratio)
        ax_row[2].imshow(foxf1_disp, cmap="magma", vmin=0.0, vmax=foxf1_vmax)
        ax_row[2].set_title(f"FOXF1 / DAPI\nthr={foxf1_threshold:.3f}")
        ax_row[2].axis("off")

        foxf1_negative_rgb = _foxf1_negative_composite(
            foxf1_ratio=foxf1_ratio,
            organoid_mask=mask,
            foxf1_positive=foxf1_positive,
            vmax=foxf1_vmax,
        )
        ax_row[3].imshow(foxf1_negative_rgb, interpolation="nearest")
        ax_row[3].set_title("FOXF1 / DAPI\nnegative=heatmap, excluded positive=red")
        ax_row[3].axis("off")

        host_disp = np.ma.masked_where(~mask, host_ratio)
        ax_row[4].imshow(host_disp, cmap="Greens")
        ax_row[4].set_title(f"Host / DAPI\nthr={host_threshold:.3f}")
        ax_row[4].axis("off")

        donor_disp = np.ma.masked_where(~mask, donor_ratio)
        ax_row[5].imshow(donor_disp, cmap="Blues")
        ax_row[5].set_title(f"Donor / DAPI\nthr={donor_threshold:.3f}")
        ax_row[5].axis("off")

        donor_halo_rgb = _donor_halo_composite(
            dapi_image=dapi_disp,
            donor_positive=donor_positive,
            donor_halo=foxf1_donor_halo,
        )
        ax_row[6].imshow(donor_halo_rgb, interpolation="nearest")
        ax_row[6].set_title(
            f"Donor graft + halo ({int(FOXF1_COMPONENT_RADIUS_PX)} px)\nblue=graft, cyan=halo"
        )
        ax_row[6].axis("off")

        ax_row[7].imshow(dapi_disp, cmap="gray")
        ax_row[7].imshow(
            np.ma.masked_where(~foxf1_positive, foxf1_positive.astype(float)),
            cmap=FOXF1_POS_CMAP,
            alpha=1.0,
            interpolation="nearest",
        )
        ax_row[7].set_title(
            "FOXF1-positive region\n"
            f"donor-adj raw={int(np.sum(foxf1_positive_raw)):,} keep={int(np.sum(foxf1_positive)):,}"
        )
        ax_row[7].axis("off")

        ax_row[8].imshow(dapi_disp, cmap="gray")
        ax_row[8].imshow(np.ma.masked_where(~host_seed, host_seed.astype(float)), cmap=HOST_POS_CMAP, alpha=0.85)
        ax_row[8].set_title(
            "Host seed component\n"
            f"thr={host_seed_threshold:.3f} keep={int(np.sum(host_seed)):,}"
        )
        ax_row[8].axis("off")

        ax_row[9].imshow(dapi_disp, cmap="gray")
        ax_row[9].imshow(np.ma.masked_where(~donor_positive, donor_positive.astype(float)), cmap=DONOR_POS_CMAP, alpha=0.85)
        ax_row[9].set_title(
            "Donor graft component\n"
            f"thr={donor_threshold:.3f} raw={int(np.sum(donor_positive_raw)):,} keep={int(np.sum(donor_positive)):,}"
        )
        ax_row[9].axis("off")

        fox_lineage = np.zeros(mask.shape + (3,), dtype=np.float32)
        fox_lineage[:] = np.repeat(dapi_disp[..., None], 3, axis=2)
        donor_rgb = np.asarray([21, 101, 192], dtype=np.float32) / 255.0
        host_rgb = np.asarray([46, 125, 50], dtype=np.float32) / 255.0
        fox_lineage[donor_partition_full] = donor_rgb
        fox_lineage[host_partition_full] = host_rgb
        fox_lineage[foxf1_positive & donor_partition_full] = 0.75 * donor_rgb + 0.25 * np.asarray([1.0, 1.0, 1.0], dtype=np.float32)
        fox_lineage[foxf1_positive & host_partition_full] = 0.75 * host_rgb + 0.25 * np.asarray([1.0, 1.0, 1.0], dtype=np.float32)
        ax_row[10].imshow(fox_lineage, interpolation="nearest")
        ax_row[10].set_title(
            "Full lineage partition\n"
            f"all host={int(np.sum(host_partition_full)):,} donor={int(np.sum(donor_partition_full)):,}\n"
            f"FOXF1 host={int(np.sum(foxf1_partition_host)):,} donor={int(np.sum(foxf1_partition_donor)):,}"
        )
        ax_row[10].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_classification_debug(representative_planes)

            ## FOXF1 Threshold Sweeps By Condition

            These panels show representative planes from each condition under a range of FOXF1 thresholds, so the graft-localized FOXF1 domain can be judged directly by eye before choosing a final condition-level cutoff.
            The red region is the donor-adjacent connected component of the thresholded FOXF1 mask, not the full raw thresholded FOXF1 mask.
            

In [ ]:
FOXF1_SWEEP_SCALES = [0.75, 1.00, 1.25, 1.50]


def foxf1_sweep_thresholds_for_condition(condition_name: str) -> list[float]:
    channel_rows = threshold_df[
        (threshold_df["channel_key"].astype(str) == "foxf1")
        & (threshold_df["threshold_group_condition"].astype(str) == str(condition_name))
    ].copy()
    if channel_rows.empty:
        return [0.02, 0.04, 0.06, 0.08]
    center = float(channel_rows["threshold_value"].median())
    center = max(center, 1.0e-6)
    return [float(max(0.0, center * scale)) for scale in FOXF1_SWEEP_SCALES]


def plot_condition_foxf1_threshold_sweep(
    rep_df: pd.DataFrame,
    condition_name: str,
    sweep_thresholds: list[float],
) -> None:
    condition_df = rep_df[rep_df["condition"].astype(str) == str(condition_name)].copy().sort_values(["file_id", "z_index"])
    if condition_df.empty:
        print(f"No representative planes available for {condition_name}.")
        return

    n_rows = len(condition_df)
    n_cols = len(sweep_thresholds)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 4.0 * n_rows))
    axes = np.asarray(axes)
    if n_rows == 1:
        axes = axes[None, :]

    for ax_row, row in zip(axes, condition_df.itertuples(index=False)):
        stack = tqh.load_transplant_stack(ROOT / str(row.file_path))
        dapi_idx = tqh.channel_index(stack.canonical_channel_names, "dapi")
        foxf1_idx = tqh.channel_index(stack.canonical_channel_names, "foxf1")

        z_index = int(row.z_index)
        dapi_raw = np.asarray(stack.data_czyx[dapi_idx, z_index], dtype=np.float32)
        foxf1_raw = np.asarray(stack.data_czyx[foxf1_idx, z_index], dtype=np.float32)
        donor_idx = tqh.channel_index(stack.canonical_channel_names, "donor")
        donor_raw = np.asarray(stack.data_czyx[donor_idx, z_index], dtype=np.float32)
        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)

        condition_name = str(row.condition)
        dapi_corrected = subtract_offmask_background(dapi_raw, organoid_mask=mask)
        foxf1_corr = subtract_offmask_background(foxf1_raw, organoid_mask=mask)
        donor_corr = subtract_offmask_background(donor_raw, organoid_mask=mask)
        dapi_floor = float(row.dapi_floor)
        foxf1_ratio = tqh.reporter_over_dapi(foxf1_corr, dapi_corrected, mask, dapi_floor=dapi_floor)
        donor_ratio = tqh.reporter_over_dapi(donor_corr, dapi_corrected, mask, dapi_floor=dapi_floor)
        donor_threshold = float(get_threshold("donor", condition=condition_name, file_id=int(row.file_id)))
        donor_positive_raw = mask & np.isfinite(donor_ratio) & (donor_ratio > donor_threshold)
        donor_positive = tqh.donor_graft_component(
            donor_positive=donor_positive_raw,
            organoid_mask=mask,
            fill_holes=True,
        )
        foxf1_vmax = float(np.nanquantile(foxf1_ratio[mask], 0.995)) if np.any(mask) else 1.0
        if not np.isfinite(foxf1_vmax) or foxf1_vmax <= 0:
            foxf1_vmax = 1.0
        foxf1_disp = np.ma.masked_where(~mask, foxf1_ratio)

        for ax, sweep_threshold in zip(ax_row, sweep_thresholds):
            sweep_threshold = float(sweep_threshold)
            foxf1_positive_raw = mask & np.isfinite(foxf1_ratio) & (foxf1_ratio > sweep_threshold)
            foxf1_positive, _ = tqh.donor_adjacent_foxf1_component(
                foxf1_positive=foxf1_positive_raw,
                donor_positive=donor_positive,
                organoid_mask=mask,
                radius_px=FOXF1_COMPONENT_RADIUS_PX,
            )
            ax.imshow(foxf1_disp, cmap="magma", vmin=0.0, vmax=foxf1_vmax)
            ax.imshow(
                np.ma.masked_where(~foxf1_positive, foxf1_positive.astype(float)),
                cmap=FOXF1_POS_CMAP,
                alpha=1.0,
                interpolation="nearest",
            )
            ax.set_title(
                f"{row.position_label} | z{z_index}\n"
                f"thr={sweep_threshold:.3f}\n"
                f"raw={int(np.sum(foxf1_positive_raw)):,} keep={int(np.sum(foxf1_positive)):,}"
            )
            ax.axis("off")

    condition_title = str(condition_name).replace("_", " ")
    fig.suptitle(f"{condition_title} FOXF1 threshold sweep", y=1.01, fontsize=16)
    plt.tight_layout()
    plt.show()


for condition_name in ["ctrl", "BMPR1A_KD", "LDN"]:
    print(f"\n{condition_name} threshold sweep")
    sweep_thresholds = foxf1_sweep_thresholds_for_condition(condition_name)
    plot_condition_foxf1_threshold_sweep(
        representative_planes,
        condition_name=condition_name,
        sweep_thresholds=list(sweep_thresholds),
    )

            ## Interpretation Note

            This notebook is still a first-pass class-based quantification.
            If the manuscript claim depends on *host FOXF1 specifically near the donor interface*, the next iteration should add a distance-to-donor or interface-zone analysis rather than relying only on whole-host averages.
            